# 🧪 Tox21 Drug Toxicity — Exploratory Data Analysis

This notebook explores the Tox21 dataset:
- Dataset overview and class distributions
- SMILES validity and dataset quality
- Molecular descriptor distributions
- Correlation between descriptors and toxicity

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_processing import load_tox21, clean_dataset, TOX21_TARGETS
from feature_engineering import compute_molecular_descriptors, SELECTED_DESCRIPTORS

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

## 1. Load & Clean Data

In [ ]:
df = load_tox21('../data/tox21.csv')
df = clean_dataset(df)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Class Distribution per Assay Target

In [ ]:
available_targets = [t for t in TOX21_TARGETS if t in df.columns]

# Count positives, negatives, NaN per target
stats = []
for t in available_targets:
    col = df[t].dropna()
    pos = (col == 1).sum()
    neg = (col == 0).sum()
    nan = df[t].isna().sum()
    stats.append({'target': t, 'positive': pos, 'negative': neg, 'missing': nan,
                  'pos_rate': pos / (pos + neg) * 100})

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, t in enumerate(available_targets):
    ax = axes[i]
    col = df[t].dropna()
    counts = col.value_counts()
    ax.bar(['Non-toxic (0)', 'Toxic (1)'], [counts.get(0, 0), counts.get(1, 0)],
           color=['#2a9d8f', '#e63946'])
    ax.set_title(t, fontsize=11)
    ax.set_ylabel('Count')

for j in range(len(available_targets), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Class Distribution per Tox21 Assay', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/class_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. SMILES Length Distribution

In [ ]:
df['smiles_len'] = df['smiles'].str.len()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['smiles_len'], bins=60, color='#457b9d', edgecolor='white')
ax.set_xlabel('SMILES String Length')
ax.set_ylabel('Count')
ax.set_title('Distribution of SMILES String Lengths')
plt.tight_layout()
plt.show()

print(df['smiles_len'].describe())

## 4. Molecular Descriptor Analysis

In [ ]:
# Compute descriptors for a subset (for speed)
SAMPLE_N = 2000
sample_smiles = df['smiles'].sample(min(SAMPLE_N, len(df)), random_state=42)

desc_df, valid_mask = compute_molecular_descriptors(sample_smiles, show_progress=True)
print(f'Valid molecules: {valid_mask.sum()} / {len(sample_smiles)}')
desc_df.head()

In [ ]:
# Plot distributions of key descriptors
key_descs = ['MolWt', 'MolLogP', 'TPSA', 'NumAromaticRings', 'NumHDonors', 'NumHAcceptors']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, desc in enumerate(key_descs):
    ax = axes[i]
    data = desc_df[desc].dropna()
    ax.hist(data, bins=50, color='#2a9d8f', edgecolor='white', alpha=0.85)
    ax.set_title(desc)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='red', linestyle='--', lw=1.5, label=f'Median: {data.median():.2f}')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Key Molecular Descriptors', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/descriptor_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Lipinski's Rule of Five Analysis

In [ ]:
# Lipinski Rule of 5 compliance
ro5 = pd.DataFrame()
ro5['MW_ok'] = desc_df['MolWt'] <= 500
ro5['LogP_ok'] = desc_df['MolLogP'] <= 5
ro5['HBD_ok'] = desc_df['NumHDonors'] <= 5
ro5['HBA_ok'] = desc_df['NumHAcceptors'] <= 10
ro5['violations'] = (~ro5).sum(axis=1)
ro5['drug_like'] = ro5['violations'] <= 1

drug_like_pct = ro5['drug_like'].mean() * 100
print(f'Drug-like compounds (≤1 Ro5 violation): {drug_like_pct:.1f}%')

fig, ax = plt.subplots(figsize=(7, 4))
ro5['violations'].value_counts().sort_index().plot(kind='bar', ax=ax,
    color='#e76f51', edgecolor='white')
ax.set_xlabel('Number of Ro5 Violations')
ax.set_ylabel('Count')
ax.set_title("Lipinski's Rule of Five — Violation Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../results/lipinski_ro5.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Correlation Heatmap — Molecular Descriptors

In [ ]:
corr_descs = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors',
              'NumAromaticRings', 'RingCount', 'NumRotatableBonds', 'FractionCSP3']

corr = desc_df[corr_descs].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            mask=mask, vmin=-1, vmax=1, square=True, linewidths=0.5)
ax.set_title('Molecular Descriptor Correlation Matrix')
plt.tight_layout()
plt.savefig('../results/descriptor_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Descriptor vs Toxicity (SR-MMP target)

In [ ]:
from data_processing import get_binary_target

target = 'SR-MMP'
if target in df.columns:
    smiles_t, labels_t = get_binary_target(df, target)
    
    # Compute descriptors for this subset
    desc_t, _ = compute_molecular_descriptors(smiles_t, show_progress=True)
    desc_t['toxic'] = labels_t.values

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, feat in enumerate(['MolWt', 'MolLogP', 'TPSA']):
        ax = axes[i]
        for label, color, name in [(0, '#2a9d8f', 'Non-toxic'), (1, '#e63946', 'Toxic')]:
            data = desc_t[desc_t['toxic'] == label][feat].dropna()
            ax.hist(data, bins=40, alpha=0.65, color=color, label=name, edgecolor='white')
        ax.set_title(f'{feat} — {target}')
        ax.set_xlabel(feat)
        ax.legend()
    plt.tight_layout()
    plt.savefig('../results/descriptor_vs_toxicity.png', dpi=120, bbox_inches='tight')
    plt.show()